In [52]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/dejintime/moviereviews/testData.tsv/testData.tsv
/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv
/kaggle/input/datasets/dejintime/moviereviews/unlabeledTrainData.tsv/unlabeledTrainData.tsv


In [53]:
train_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/labeledTrainData.tsv/labeledTrainData.tsv",
                        header = 0,
                        delimiter = "\t",
                        quoting = 3)
unlabeledTrain_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/unlabeledTrainData.tsv/unlabeledTrainData.tsv",
                                 header = 0,
                                 delimiter = "\t",
                                 quoting = 3)
test_data = pd.read_csv("/kaggle/input/datasets/dejintime/moviereviews/testData.tsv/testData.tsv",
                       header = 0,
                       delimiter = "\t",
                       quoting = 3)
print(f'the nums of train reviews:{train_data["review"].size}\nthe nums of unlabeled_train reviews:{unlabeledTrain_data["review"].size}\nthe nums of test_data reviews:{test_data["review"].size}\n')

the nums of train reviews:25000
the nums of unlabeled_train reviews:50000
the nums of test_data reviews:25000



In [66]:
from bs4 import BeautifulSoup
import re
from nltk.corpus import stopwords
def CleanData_to_wordlist(raw_review, remove_stopwords=False):
    review_text = BeautifulSoup(raw_review).get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    if remove_stopwords:
        words = [w for w in words if w not in stopwords.words("english")]
    return words

In [67]:
import nltk.data
nltk.download("punkt")
tokenizer = nltk.data.load('tokenizers/punkt/english.pickle')
def review_to_sentences(review, tokenizer, remove_stopwords = False):
    raw_sentences = tokenizer.tokenize(review.strip())
    sentences = []
    for raw_sentence in raw_sentences:
        if len(raw_sentence) > 0:
            sentences.append(CleanData_to_wordlist(raw_sentence, remove_stopwords))
    return sentences

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [68]:
sentences = []
print(f"parsing sentences from training set\n")
for review in train_data["review"]:
    sentences += review_to_sentences(review, tokenizer)
print(f"parsing sentences from unlabeled_training set\n")
for review in unlabeledTrain_data["review"]:
    sentences += review_to_sentences(review, tokenizer)

parsing sentences from training set



/tmp/ipykernel_55/394599830.py:5: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a URL than HTML or XML.

If you meant to use Beautiful Soup to parse the web page found at a certain URL, then something has gone wrong. You should use an Python package like 'requests' to fetch the content behind the URL. Once you have the content as a string, you can feed that string into Beautiful Soup.

However, if you want to parse some data that happens to look like a URL, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  review_text = BeautifulSoup(raw_review).get_text()


parsing sentences from unlabeled_training set



In [70]:
len(sentences)

796172

In [72]:
print(sentences[0])

['with', 'all', 'this', 'stuff', 'going', 'down', 'at', 'the', 'moment', 'with', 'mj', 'i', 've', 'started', 'listening', 'to', 'his', 'music', 'watching', 'the', 'odd', 'documentary', 'here', 'and', 'there', 'watched', 'the', 'wiz', 'and', 'watched', 'moonwalker', 'again']


In [75]:
import logging
logging.basicConfig(format='%(asctime)s:%(levelname)s:%(message)s',level = logging.INFO)
num_features = 300
min_word_count = 40
num_workers = 4
context = 10
downsampling = 1e-3
from gensim.models import word2vec
print("training model...\n")
model = word2vec.Word2Vec(sentences, workers = num_workers, vector_size = num_features,min_count = min_word_count, window = context, sample = downsampling)
model.init_sims(replace=True)
model_name = "300features_40minwords_10context"
model.save(model_name)

2026-04-28 08:41:31,727:INFO:collecting all words and their counts
2026-04-28 08:41:31,729:INFO:PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-04-28 08:41:31,784:INFO:PROGRESS: at sentence #10000, processed 225664 words, keeping 17775 word types
2026-04-28 08:41:31,841:INFO:PROGRESS: at sentence #20000, processed 451738 words, keeping 24945 word types
2026-04-28 08:41:31,897:INFO:PROGRESS: at sentence #30000, processed 670859 words, keeping 30027 word types


training model...



2026-04-28 08:41:31,955:INFO:PROGRESS: at sentence #40000, processed 896841 words, keeping 34335 word types
2026-04-28 08:41:32,015:INFO:PROGRESS: at sentence #50000, processed 1116082 words, keeping 37751 word types
2026-04-28 08:41:32,082:INFO:PROGRESS: at sentence #60000, processed 1337544 words, keeping 40711 word types
2026-04-28 08:41:32,160:INFO:PROGRESS: at sentence #70000, processed 1560307 words, keeping 43311 word types
2026-04-28 08:41:32,224:INFO:PROGRESS: at sentence #80000, processed 1779516 words, keeping 45707 word types
2026-04-28 08:41:32,285:INFO:PROGRESS: at sentence #90000, processed 2003714 words, keeping 48121 word types
2026-04-28 08:41:32,348:INFO:PROGRESS: at sentence #100000, processed 2225465 words, keeping 50190 word types
2026-04-28 08:41:32,407:INFO:PROGRESS: at sentence #110000, processed 2444323 words, keeping 52058 word types
2026-04-28 08:41:32,466:INFO:PROGRESS: at sentence #120000, processed 2666488 words, keeping 54098 word types
2026-04-28 08:41:

In [79]:
model.wv.most_similar("man")

[('woman', 0.6065600514411926),
 ('lady', 0.5857177972793579),
 ('lad', 0.5814040899276733),
 ('men', 0.5237018465995789),
 ('soldier', 0.5164439678192139),
 ('guy', 0.515480637550354),
 ('millionaire', 0.5146815180778503),
 ('monk', 0.5092204809188843),
 ('farmer', 0.5044004917144775),
 ('businessman', 0.5014628767967224)]

In [80]:
model.wv.most_similar("awful")

[('terrible', 0.749895453453064),
 ('atrocious', 0.716654896736145),
 ('abysmal', 0.7113090753555298),
 ('horrible', 0.7085697650909424),
 ('dreadful', 0.7079139947891235),
 ('horrendous', 0.663122296333313),
 ('appalling', 0.6534457206726074),
 ('horrid', 0.6431335806846619),
 ('amateurish', 0.6057510375976562),
 ('lousy', 0.5992722511291504)]